<table style="border:0;width:100%;border-collapse:collapse"><tr style="border:0"><td style="border:0;width:105px;vertical-align:middle"><img src="../Imagenes/logo-final-ap.png" width="90"/></td><td style="border:0;vertical-align:middle"><span style="font-size:32px;font-weight:700;color:#bd232b">Aprendizaje Profundo</span></td></tr></table>

[https://github.com/AprendizajeProfundo](https://github.com/AprendizajeProfundo)

# <span style="color:#174ea6">Cuaderno 103A · Actor-Crítico y PPO: aprender a decidir sin cambiar demasiado deprisa</span>

**Sesión 2 · De aprender valores a aprender políticas; preparación para PPO y Hydro-Lab**

**Autores:** Alvaro Montenegro · Daniel Montenegro · Pilar Montenegro  
**Docente:** Alvaro Montenegro, [ammontenegrod@gmail.com](mailto:ammontenegrod@gmail.com),  
**Institución:** Universidad de los Andes - Ecopetrol  
**Edición docente:** 103 · Octubre de 2026  
**Licencia del contenido y código originales de esta edición:** MIT (véase sección final).

> **Pregunta rectora.** ¿Cómo optimizar directamente una política estocástica y por qué estimar una función de valor puede reducir la variabilidad de su aprendizaje?


---

**Generación documental:** G1.0 · 26 de octubre de 2026  
**Ubicación:** entre los cuadernos 103 y 104  
**Modalidad:** lección conceptual con verificaciones numéricas

## Objetivos de aprendizaje

Al finalizar esta lección, el estudiante podrá explicar por qué resulta útil separar la selección de acciones de la evaluación de los estados, cómo funcionan conjuntamente el Actor y el Crítico y por qué PPO introduce restricciones sobre los cambios de política.

También comprenderá cómo se generan las experiencias, cómo se calculan los retornos y las ventajas, cómo se construye la función objetivo de PPO y cómo se entrenan las redes neuronales.

La pregunta que guiará toda la lección es:

**¿Cómo puede un agente aprender a tomar mejores decisiones sin destruir, con cada actualización, lo que ya había aprendido?**

---

# 1. Nuestro robot tiene un nuevo problema

Recordemos al pequeño robot de la lección anterior.

Su misión continúa siendo encontrar una estación de recarga en un laberinto.

En Deep Q-Learning aprendimos a estimar cuánto valía ejecutar cada acción en una situación determinada.

La red neuronal recibía un estado y producía valores como estos:

| Acción | Valor Q |
|---|---:|
| Avanzar | 8 |
| Girar a la izquierda | 5 |
| Girar a la derecha | 2 |

El agente podía seleccionar la acción con mayor valor Q, aunque ocasionalmente exploraba otras alternativas.

Ahora vamos a cambiar la manera de aprender.

En lugar de pedirle a una red que calcule directamente el valor de cada acción, construiremos un sistema con dos responsabilidades diferentes.

Una parte decidirá qué hacer.

Otra evaluará qué tan favorable es la situación en la que se encuentra el agente.

A la primera la llamaremos **Actor**.

A la segunda la llamaremos **Crítico**.

Y veremos cómo PPO utiliza ambos componentes para mejorar gradualmente el comportamiento.


<img src="../Imagenes/Infografia_Actor_Critico_Accion.png" width="800"/>

*Figura 103A.1.  Las dos responsabilidades del sistema: decidir y evaluar.*


---

# 2. El Actor: quien toma las decisiones

Imaginemos que nuestro robot se encuentra frente a una bifurcación.

Puede avanzar, girar a la izquierda o girar a la derecha.

En lugar de asignar un valor Q a cada acción, el Actor produce una distribución de probabilidad.

Por ejemplo:

| Acción | Probabilidad |
|---|---:|
| Avanzar | 0.10 |
| Izquierda | 0.70 |
| Derecha | 0.20 |
| **Total** | **1.00** |

El Actor considera que girar a la izquierda es la alternativa más prometedora.

Pero esto no significa que siempre deba seleccionarla.

Durante el entrenamiento podemos muestrear una acción de esa distribución.

Así, girar a la izquierda tendrá una probabilidad del 70 %, pero las otras acciones también podrán ejecutarse.

La distribución se denomina **política**.

La representamos mediante:

$$
\pi_\theta(a\mid s)
$$

Donde:

- $s$ representa el estado.
- $a$ representa una acción.
- $\theta$ representa los parámetros de la red neuronal del Actor.
- $\pi_\theta(a\mid s)$ representa la probabilidad de seleccionar la acción $a$ en el estado $s$.

El Actor aprende modificando sus parámetros para aumentar o disminuir las probabilidades de determinadas acciones.

Esta es una diferencia fundamental respecto de DQN.

**DQN aprende una función de valor de acciones. El Actor aprende directamente una política.**


<img src="../Imagenes/Politica_probabilistica_del_robot_en_accion.png" width="800"/>

*Figura 103A.2.  Distinguir una política probabilística de una regla determinista.*


---

# 3. El Crítico: quien evalúa las situaciones

Ahora imaginemos que el robot llega a una nueva posición.

¿Cómo sabemos si esa posición es favorable?

El Crítico intenta responder esa pregunta.

En una arquitectura Actor-Crítico habitual, el Crítico aproxima una función de valor de estado:

$$
V_\phi(s)
$$

Esta función representa el retorno descontado esperado al comenzar en el estado $s$ y continuar siguiendo la política actual.

El símbolo $\phi$ representa los parámetros del Crítico.

Supongamos que estima:

| Estado | Valor estimado |
|---|---:|
| A | 3 |
| B | 8 |
| C | 15 |
| D | −5 |

El Crítico considera que C es una situación especialmente favorable.

Pero debemos tener cuidado con la interpretación.

Un valor elevado no significa necesariamente que el robot esté físicamente cerca de la meta.

Significa que, bajo la política actual y las recompensas definidas, se espera un retorno descontado relativamente elevado.

El Crítico tampoco proporciona necesariamente la acción correcta.

Su función principal es evaluar las situaciones y aportar una señal útil para entrenar al Actor.


<img src="../Imagenes/El_Critico_y_los_valores_de_estado.png" width="800"/>

*Figura 103A.3.  El Crítico estima valores de estados, no instrucciones directas de movimiento.*

---

# 4. ¿Cómo trabajan juntos?

Tenemos dos redes neuronales.

El Actor recibe el estado y produce probabilidades de acciones.

El Crítico recibe el estado y produce una estimación de valor.

Podemos representar el sistema mediante:

$$
s_t\longrightarrow
\begin{cases}
\pi_\theta(\cdot\mid s_t)&\text{Actor}\\
V_\phi(s_t)&\text{Crítico}
\end{cases}
$$

El Actor selecciona una acción.

El ambiente responde con una recompensa y un nuevo estado.

El Crítico ayuda a determinar si la experiencia fue mejor o peor de lo esperado.

Esa evaluación se utiliza para modificar la política.

Conviene aclarar algo: el Crítico no tiene que enviarle una orden al Actor durante cada decisión. Su papel principal aparece en el entrenamiento.

En una implementación podemos utilizar dos redes completamente separadas o una arquitectura que comparta capas y tenga dos cabezas de salida.


<img src="../Imagenes/Dialogo_Actor_Critico.png" width="800"/>

*Figura 103A.4.  Cómo se relacionan decisión, experiencia y evaluación.*

---

# 5. La idea de ventaja: ¿fue mejor de lo esperado?

Este es uno de los conceptos más importantes de Actor-Crítico.

Supongamos que nuestro robot ejecuta una acción y obtiene un resultado favorable.

¿Debemos aumentar la probabilidad de esa acción?

La respuesta depende de qué tan favorable era el resultado en comparación con lo esperado.

Consideremos dos situaciones.

En la primera, el robot obtiene un retorno de 10, pero el Crítico esperaba 15.

En la segunda, obtiene un retorno de 8, pero el Crítico esperaba 3.

Aunque el retorno de la primera situación es mayor, la segunda supera más ampliamente las expectativas.

Esta diferencia se expresa mediante la **ventaja**, conocida como *Advantage*.

Conceptualmente:

$$
A^\pi(s,a)=Q^\pi(s,a)-V^\pi(s)
$$

La ventaja compara el valor esperado de ejecutar una acción con el valor promedio esperado del estado bajo la política.

- Ventaja positiva: la acción es mejor que el promedio de la política en ese estado.
- Ventaja negativa: la acción es peor.
- Ventaja cercana a cero: no hay una diferencia importante.

En la práctica, no conocemos exactamente estas funciones y debemos estimarlas a partir de experiencias.


<img src="../Imagenes/Comparacion_ventajas_en_aprendizaje reforzado.png" width="800"/>

*Figura 103A.5.  Una recompensa o retorno elevado no implica necesariamente una ventaja positiva.*

---

# 6. El error de diferencia temporal

¿Cómo estimamos si una experiencia fue mejor de lo esperado?

Podemos utilizar una expresión que ya conocemos de la lección de DQN.

El error de diferencia temporal es:

$$
\delta_t=
r_t+\gamma V_\phi(s_{t+1})-V_\phi(s_t)
$$

Para una transición terminal, eliminamos el término de valor futuro.

Esta expresión compara:

**Lo que ahora estimamos que produjo la transición**, es decir, la recompensa inmediata más el valor descontado del siguiente estado,

con:

**Lo que el Crítico esperaba antes de actuar**, es decir, el valor del estado inicial.

Veamos un ejemplo.

Supongamos:

$$
r_t=2
$$

$$
V(s_t)=5
$$

$$
V(s_{t+1})=8
$$

$$
\gamma=0.9
$$

Entonces:

$$
\delta_t=2+0.9(8)-5
$$

$$
\boxed{\delta_t=4.2}
$$

La transición resultó mejor de lo esperado según las estimaciones actuales del Crítico.

Este error puede utilizarse como una estimación de ventaja de un paso.

Pero PPO suele emplear una técnica más elaborada, que estudiaremos enseguida.


<img src="../Imagenes/Error_temporal_TD.png" width="800"/>

*Figura 103A.6.  Una transición produce una señal para evaluar la acción.*


---

# 7. ¿Por qué no utilizar únicamente una transición?

Supongamos que el robot ejecuta una buena acción, pero la recompensa importante llega cinco pasos después.

Si solo examinamos la recompensa inmediata, podríamos no reconocer adecuadamente el mérito de esa acción.

Podemos utilizar retornos de varios pasos para incorporar más información futura.

Sin embargo, aparece un compromiso.

Las estimaciones que dependen mucho del Crítico pueden introducir sesgo cuando sus valores son incorrectos.

Las estimaciones que dependen de retornos largos pueden presentar una varianza elevada.

Para equilibrar estas propiedades, PPO suele utilizar **Generalized Advantage Estimation (GAE)**.

La idea consiste en combinar errores de diferencia temporal de distintos horizontes.

Una expresión habitual es:

$$
\hat A_t^{GAE(\gamma,\lambda)}
=
\sum_{l=0}^{T-t-1}
(\gamma\lambda)^l\delta_{t+l}
$$

Para simplificar, suponemos aquí que el tramo considerado no cruza una terminación real del episodio; en la implementación deben utilizarse las máscaras correspondientes.

El parámetro $\lambda$ controla cuánto peso damos a los errores futuros.

Cuando $\lambda=0$, obtenemos la estimación de un paso.

Cuando $\lambda$ se aproxima a 1, incorporamos más información de pasos posteriores.

GAE no elimina completamente el sesgo ni la varianza, pero permite regular su equilibrio.

### Ejemplo numérico

Supongamos tres errores consecutivos:

$$
\delta_0=2,\qquad
\delta_1=1,\qquad
\delta_2=-1
$$

Utilizamos:

$$
\gamma=0.9,\qquad\lambda=0.8
$$

Entonces:

$$
\hat A_0=
2+(0.9)(0.8)(1)+(0.9\cdot0.8)^2(-1)
$$

$$
\hat A_0=2+0.72-0.5184
$$

$$
\boxed{\hat A_0=2.2016}
$$

Esta es una estimación de ventaja que incorpora información de varias transiciones.


<img src="../Imagenes/ventaja_generalizada.png" width="800"/>

*Figura 103A.7.  Acumulación descontada de errores temporales.*


---

In [ ]:
# Verificación del ejemplo de GAE de la sección 7
gamma, lam = 0.9, 0.8
deltas = [2, 1, -1]
ventaja = sum((gamma * lam) ** k * delta for k, delta in enumerate(deltas))
print(f"Ventaja GAE: {ventaja:.4f}")
assert abs(ventaja - 2.2016) < 1e-10

# 8. El problema de actualizar demasiado la política

Ya sabemos cómo construir señales que indican si determinadas acciones fueron mejores o peores de lo esperado.

Podríamos utilizarlas para modificar la política del Actor.

Pero aparece un problema.

Supongamos que la política actual asigna estas probabilidades:

| Acción | Probabilidad |
|---|---:|
| Izquierda | 0.60 |
| Derecha | 0.30 |
| Avanzar | 0.10 |

Después de una experiencia favorable al girar a la izquierda, una actualización excesivamente grande podría producir:

| Acción | Probabilidad nueva |
|---|---:|
| Izquierda | 0.99 |
| Derecha | 0.005 |
| Avanzar | 0.005 |

El agente prácticamente ha abandonado la exploración de otras posibilidades.

Peor aún, la evidencia que motivó el cambio puede ser insuficiente o ruidosa.

Una actualización muy grande puede deteriorar el rendimiento de la política.

**PPO intenta evitar que la optimización modifique la política de manera excesivamente agresiva.**

Su nombre completo es *Proximal Policy Optimization*.

La palabra *proximal* expresa precisamente la idea de mantener las actualizaciones cerca de una política de referencia.


<img src="../Imagenes/Actualizaciones_moderadas_aprendizaje_po_rrefuerzo.png" width="800"/>

*Figura 103A.8.  Por qué una mejora local aparente no justifica necesariamente un cambio radical de política.*

---

# 9. La idea central de PPO: comparar la política nueva con la anterior

PPO utiliza un cociente de probabilidades.

Supongamos que el agente ejecutó la acción Izquierda bajo una política anterior.

Esa política asignaba:

$$
\pi_{\theta_{\mathrm{old}}}(\text{Izquierda}\mid s)=0.40
$$

Después de modificar los parámetros, la nueva política asigna:

$$
\pi_\theta(\text{Izquierda}\mid s)=0.60
$$

Definimos:

$$
r_t(\theta)=
\frac{\pi_\theta(a_t\mid s_t)}
{\pi_{\theta_{\mathrm{old}}}(a_t\mid s_t)}
$$

En nuestro ejemplo:

$$
r_t(\theta)=\frac{0.60}{0.40}=1.5
$$

La probabilidad de la acción observada aumentó un 50 % respecto de su valor anterior.

Si el cociente es 1, la probabilidad no ha cambiado.

Si es mayor que 1, ha aumentado.

Si es menor que 1, ha disminuido.

Es importante distinguir este cociente $r_t(\theta)$ de la recompensa del ambiente, que también suele representarse mediante la letra $r$.

Aquí estamos comparando probabilidades, no recompensas.


<img src="../Imagenes/Comparacion_visual _probabilidades_en_PPO.png" width="800"/>

*Figura 103A.9.  Cómo PPO cuantifica cambios en la probabilidad de una acción observada.*


---

# 10. El clipping: el corazón de PPO

PPO introduce un mecanismo denominado **clipping** o recorte.

Su objetivo es limitar el incentivo de optimización cuando el cociente de probabilidades se aleja demasiado de 1.

Definimos un parámetro:

$$
\varepsilon>0
$$

Por ejemplo:

$$
\varepsilon=0.2
$$

El intervalo de referencia será:

$$
[1-\varepsilon,1+\varepsilon]=[0.8,1.2]
$$

La función objetivo recortada de PPO es:

$$
L^{CLIP}(\theta)=
\mathbb E_t\left[
\min\left(
r_t(\theta)\hat A_t,
\operatorname{clip}(r_t(\theta),1-\varepsilon,1+\varepsilon)\hat A_t
\right)
\right]
$$

Vamos a entenderla sin dejarnos intimidar por la fórmula.

Primero calculamos cuánto cambió la probabilidad de la acción.

Después multiplicamos ese cambio por la ventaja estimada.

Finalmente, comparamos el resultado con una versión en la que el cociente se recorta al intervalo permitido.

Seleccionamos el menor de los dos términos.

Esto limita el beneficio que el objetivo puede atribuir a determinados cambios excesivos de probabilidad.

**Una precisión importante:** PPO no impone una restricción absoluta que prohíba que las probabilidades salgan del intervalo. El clipping modifica la función objetivo y desincentiva ciertos cambios, pero no garantiza por sí solo que todas las actualizaciones permanezcan dentro de esos límites.



<img src="../Imagenes/clipping_PPO.png" width="800"/>

*Figura 103A.10.  El efecto del recorte sobre la función objetivo.*

---

# 11. Un ejemplo completo del clipping

Supongamos que una acción produjo una ventaja positiva:

$$
\hat A_t=4
$$

La política anterior asignaba probabilidad 0.20 y la nueva asigna 0.30.

Entonces:

$$
r_t=\frac{0.30}{0.20}=1.5
$$

Con $\varepsilon=0.2$, el cociente recortado es:

$$
\operatorname{clip}(1.5,0.8,1.2)=1.2
$$

Calculamos los dos términos:

$$
1.5(4)=6
$$

$$
1.2(4)=4.8
$$

PPO utiliza:

$$
\boxed{\min(6,4.8)=4.8}
$$

Ahora examinemos una ventaja negativa:

$$
\hat A_t=-4
$$

Supongamos que la probabilidad de la acción se redujo a la mitad:

$$
r_t=0.5
$$

El cociente recortado es 0.8.

Calculamos:

$$
0.5(-4)=-2
$$

$$
0.8(-4)=-3.2
$$

Por tanto:

$$
\boxed{\min(-2,-3.2)=-3.2}
$$

Observemos que el comportamiento del clipping depende del signo de la ventaja.

Con ventaja positiva, el recorte limita el incentivo de aumentar demasiado la probabilidad de la acción.

Con ventaja negativa, limita el incentivo de disminuirla demasiado.

La función está diseñada para desalentar determinadas modificaciones excesivas, sin convertir la actualización en una regla rígida.


<img src="../Imagenes/ventajas positiva_y_negativa.png" width="800"/>

*Figura 103A.11.  El clipping actúa de manera diferente según el signo de la ventaja.*


---

In [ ]:
# Ejemplos de clipping para ventajas de signo positivo y negativo
def objetivo_ppo(prob_nueva, prob_anterior, ventaja, epsilon=0.2):
    ratio = prob_nueva / prob_anterior
    ratio_recortado = min(max(ratio, 1 - epsilon), 1 + epsilon)
    return min(ratio * ventaja, ratio_recortado * ventaja)

positivo = objetivo_ppo(0.30, 0.20, 4)
negativo = objetivo_ppo(0.10, 0.20, -4)
print(f"Ventaja positiva: {positivo:.1f}; negativa: {negativo:.1f}")
assert positivo == 4.8
assert negativo == -3.2

# 12. ¿Cómo se generan los datos para entrenar PPO?

Regresemos a nuestro robot.

En DQN almacenábamos transiciones en un Replay Buffer y podíamos reutilizar experiencias antiguas.

PPO funciona de otra manera.

Primero utilizamos la política actual para interactuar con el ambiente durante un número determinado de pasos.

Por ejemplo, ejecutamos 2.048 interacciones, posiblemente distribuidas entre varios ambientes paralelos.

Durante la recolección almacenamos información como:

$$
(s_t,a_t,r_t,s_{t+1},d_t,\log\pi_{\mathrm{old}}(a_t\mid s_t),V(s_t))
$$

El registro puede incluir información adicional necesaria para calcular los retornos, las ventajas y las condiciones de terminación.

A este conjunto temporal de experiencias lo llamaremos **rollout buffer**.

No es equivalente al Replay Buffer de DQN.

PPO es un algoritmo *on-policy*: utiliza datos generados por la política de referencia actual y los emplea para realizar un número limitado de actualizaciones.

Después recolecta nuevas experiencias con la política actualizada.

En PPO convencional no mantenemos indefinidamente experiencias antiguas para seguir entrenando la política.


<img src="../Imagenes/Flujo_PPO_del_buffer_de_rollout.png" width="800"/>

*Figura 103A.12.  Distinguir el almacenamiento temporal de PPO del Replay Buffer reutilizable de DQN.*


---

# 13. ¿Cómo aprende el Crítico?

Hasta ahora hemos concentrado nuestra atención en el Actor.

Pero el Crítico también debe entrenarse.

Su objetivo es aproximar el retorno esperado desde cada estado.

Para ello necesitamos construir objetivos de valor.

Una opción es utilizar retornos descontados obtenidos de las experiencias.

Otra opción habitual, cuando utilizamos GAE, consiste en construir:

$$
\hat R_t=\hat A_t+V_{\phi_{\mathrm{old}}}(s_t)
$$

Aquí, el valor utilizado para construir el objetivo corresponde a la estimación de referencia calculada durante la recolección.

El Crítico actual produce:

$$
V_\phi(s_t)
$$

Podemos entrenarlo minimizando:

$$
L_V(\phi)=
\frac{1}{N}\sum_{t=1}^{N}
\left(V_\phi(s_t)-\hat R_t\right)^2
$$

Es una función de pérdida cuadrática, semejante a las utilizadas en aprendizaje supervisado.

Supongamos:

$$
\hat R_t=12
$$

$$
V_\phi(s_t)=9
$$

Entonces:

$$
L_V=(9-12)^2=9
$$

La retropropagación modifica los parámetros del Crítico para reducir la discrepancia.

De esta manera, el Actor aprende a modificar las probabilidades de las acciones y el Crítico aprende a mejorar sus estimaciones de valor.


<img src="../Imagenes/Entrenamiento_del_Critico.png" width="800"/>

*Figura 103A.13.  El Crítico también es una red entrenable.*


---

# 14. La entropía: evitar decisiones prematuramente rígidas

Hay otro elemento importante en muchas implementaciones de PPO.

Recordemos que el Actor produce una distribución de probabilidad sobre las acciones.

Al principio, puede resultar conveniente que mantenga cierta diversidad.

Si asigna probabilidad casi 1 a una sola acción demasiado pronto, podría dejar de explorar alternativas prometedoras.

Una manera de medir la dispersión de una política discreta es mediante la entropía:

$$
H(\pi(\cdot\mid s))=
-\sum_a\pi(a\mid s)\log\pi(a\mid s)
$$

Una distribución uniforme tiene mayor entropía que una distribución muy concentrada.

Por ejemplo:

$$
\pi_1=(0.33,0.33,0.34)
$$

es más dispersa que:

$$
\pi_2=(0.98,0.01,0.01)
$$

Algunas implementaciones añaden una bonificación de entropía al objetivo de entrenamiento para favorecer la exploración.

Sin embargo, esta bonificación es opcional y su intensidad debe ajustarse cuidadosamente.

Demasiada entropía puede impedir que la política consolide decisiones eficaces.



<img src="../Imagenes/Entropia_exploracion_en_aprendizaje_reforzado.png" width="800"/>

*Figura 103A.14.  La concentración de probabilidades influye en la diversidad de acciones.*

---

# 15. La función de pérdida completa

Ahora podemos reunir las piezas.

El Actor busca maximizar el objetivo recortado de PPO.

El Crítico busca minimizar el error de predicción de valor.

Opcionalmente, incorporamos un término de entropía.

Una formulación habitual para minimizar mediante descenso por gradiente es:

$$
L_{\mathrm{total}}
=
-L^{CLIP}
+c_V L_V
-c_H H
$$

Donde:

- $L^{CLIP}$ es el objetivo del Actor.
- $L_V$ es la pérdida del Crítico.
- $H$ representa la entropía promedio de la política.
- $c_V$ controla el peso de la pérdida de valor.
- $c_H$ controla la bonificación de entropía.

El signo negativo del primer término aparece porque los optimizadores suelen minimizar funciones de pérdida, mientras que queremos maximizar el objetivo de política.

Cuando Actor y Crítico tienen parámetros separados, pueden utilizarse pérdidas y optimizadores independientes. Si comparten capas, sus señales de entrenamiento pueden afectar a los parámetros compartidos.

---

# 16. El entrenamiento de PPO, paso a paso

Ya podemos describir el procedimiento completo.

**Primera fase: recolectar experiencias**

Utilizamos la política actual para interactuar con el ambiente.

Durante la interacción almacenamos estados, acciones, recompensas, probabilidades de las acciones ejecutadas, valores estimados y señales de terminación.

**Segunda fase: calcular los objetivos**

Con las experiencias recolectadas calculamos errores temporales, ventajas mediante GAE y objetivos para entrenar el Crítico.

**Tercera fase: actualizar las redes**

Dividimos los datos recolectados en minibatches.

Para cada minibatch calculamos los nuevos logaritmos de probabilidades, los cocientes de probabilidad, la función objetivo recortada y la pérdida de valor.

Actualizamos los parámetros mediante retropropagación.

Podemos recorrer varias veces los mismos datos recientes durante un número limitado de épocas.

**Cuarta fase: volver al ambiente**

Después de completar las actualizaciones, utilizamos la nueva política para generar otro conjunto de experiencias.

El ciclo se repite.


<img src="../Imagenes/Ciclo_visual_entrenamiento_PPO.png" width="800"/>

*Figura 103A.14.  Ciclo_visual_entrenamiento_PPO.*

---

# 17. Ejercicio integrador: una actualización PPO

Vamos a resolver un ejemplo que reúne los conceptos principales.

El robot ejecutó la acción Izquierda.

La política anterior asignaba:

$$
\pi_{\mathrm{old}}(\text{Izquierda}\mid s)=0.25
$$

La política nueva asigna:

$$
\pi_{\theta}(\text{Izquierda}\mid s)=0.35
$$

La ventaja estimada es:

$$
\hat A=3
$$

Utilizamos:

$$
\varepsilon=0.2
$$

**Pregunta 1.** ¿Cuál es el cociente de probabilidades?

$$
r=\frac{0.35}{0.25}=1.4
$$

**Pregunta 2.** ¿Cuál es el cociente recortado?

$$
\operatorname{clip}(1.4,0.8,1.2)=1.2
$$

**Pregunta 3.** ¿Cuál es la contribución de esta transición al objetivo PPO?

$$
r\hat A=1.4(3)=4.2
$$

$$
r_{\mathrm{clip}}\hat A=1.2(3)=3.6
$$

Por tanto:

$$
\boxed{L_t^{CLIP}=3.6}
$$

**Pregunta 4.** ¿Significa esto que PPO obligará a la probabilidad de Izquierda a permanecer por debajo de 0.30?

No.

El valor 0.30 correspondería a multiplicar la probabilidad anterior por 1.2, pero PPO no impone directamente esa restricción sobre la probabilidad.

El clipping limita la contribución al objetivo, no necesariamente el cambio efectivo de probabilidad después de cada actualización.

Esta distinción es fundamental para comprender correctamente el algoritmo.

---

In [ ]:
# Comprobación del ejercicio integrador de PPO
ratio = 0.35 / 0.25
clip_ratio = min(max(ratio, 0.8), 1.2)
contribucion = min(ratio * 3, clip_ratio * 3)
print(f"Cociente: {ratio:.1f}; cociente recortado: {clip_ratio:.1f}; objetivo: {contribucion:.1f}")
assert abs(contribucion - 3.6) < 1e-10

# 18. Actor-Crítico no es lo mismo que PPO

Conviene detenernos en una diferencia conceptual.

**Actor-Crítico es una familia de arquitecturas y métodos. PPO es un algoritmo de optimización de políticas que habitualmente utiliza una arquitectura Actor-Crítico.**

Podemos construir sistemas Actor-Crítico utilizando otros procedimientos de entrenamiento.

Por ejemplo, A2C y A3C pertenecen a esta familia.

También existen algoritmos como SAC que utilizan actores y críticos, pero con mecanismos de aprendizaje distintos.

PPO se caracteriza especialmente por su manera de optimizar la política utilizando datos recientes y un objetivo recortado.

Por tanto, no debemos utilizar los términos Actor-Crítico y PPO como sinónimos.

---

# 19. Comparación entre DQN y PPO

| Característica | DQN | PPO |
|---|---|---|
| Qué aprende directamente | Valores Q | Política y normalmente valor de estado |
| Selección de acciones | Basada en Q y exploración | Muestreo de la política |
| Tipo de aprendizaje | Off-policy | On-policy |
| Almacenamiento | Replay Buffer | Rollout Buffer |
| Reutilización de datos | Experiencias antiguas y recientes | Varias épocas sobre datos recientes |
| Acciones continuas | No directamente en DQN convencional | Sí |
| Red de referencia | Target Network | Política anterior para calcular cocientes |
| Mecanismo característico | Objetivos Bellman | Objetivo de política recortado |

Ninguno de los dos algoritmos es universalmente superior.

DQN puede aprovechar muy bien las experiencias almacenadas en problemas discretos.

PPO ofrece un procedimiento relativamente sencillo y versátil para optimizar políticas tanto discretas como continuas.

La elección depende del ambiente, el costo de obtener experiencias, el espacio de acciones y los objetivos del problema.

---

# 20. ¿Qué aprende realmente un sistema Actor-Crítico?

El Actor aprende una relación entre estados y decisiones.

El Crítico aprende una aproximación del retorno esperado desde los estados.

Ambas redes pueden construir representaciones internas útiles del ambiente.

Sin embargo, no debemos confundir esas representaciones con una comprensión completa de la dinámica del mundo.

El Crítico no descubre necesariamente las leyes físicas del ambiente.

Tampoco posee conocimiento infalible sobre el futuro.

Aprende una función de valor a partir de experiencias y objetivos estimados.

El Actor, por su parte, aprende una política que busca maximizar el retorno esperado bajo las condiciones de entrenamiento.

La calidad del comportamiento depende de la función de recompensa, de la cobertura de experiencias, de la capacidad de las redes y de la estabilidad del entrenamiento.

---

# 21. Síntesis final

Comenzamos con un robot que debía decidir cómo moverse.

En DQN aprendíamos valores Q para seleccionar acciones.

En Actor-Crítico separamos dos responsabilidades: seleccionar acciones y evaluar estados.

El Actor representa una política probabilística.

El Crítico estima valores que ayudan a construir señales de aprendizaje.

La ventaja indica si una acción resultó mejor o peor de lo esperado.

GAE permite combinar información temporal para estimar esas ventajas.

PPO compara las probabilidades de las acciones bajo la política nueva y la anterior.

Su objetivo recortado limita el incentivo de realizar determinadas modificaciones excesivas.

El Crítico se entrena para mejorar sus estimaciones de valor, mientras el Actor aprende a modificar su política.

El proceso se repite utilizando experiencias recientes.

La idea central de esta lección puede expresarse así:

**El Actor aprende qué hacer. El Crítico ayuda a evaluar las decisiones. PPO regula cómo utilizamos esa evaluación para modificar la política sin introducir cambios innecesariamente agresivos.**

---

## Para recordar

Los seis conceptos esenciales son:

- **Actor:** red que representa la política de selección de acciones.
- **Crítico:** red que estima el valor esperado de los estados.
- **Ventaja:** medida de cuánto mejor o peor resulta una acción respecto de la referencia proporcionada por la política.
- **GAE:** procedimiento para estimar ventajas combinando información de varios pasos.
- **Clipping:** mecanismo que limita el incentivo de determinadas modificaciones excesivas de la política.
- **PPO:** algoritmo que optimiza la política utilizando ventajas estimadas y actualizaciones controladas.

## Próxima lección

**PPO por dentro: de las experiencias a los gradientes.**

Construiremos un ejemplo completo con un pequeño ambiente, dos redes neuronales y un minibatch de experiencias.

Calcularemos manualmente las probabilidades de las acciones, los errores temporales, las ventajas GAE, los cocientes de probabilidades y la función de pérdida.

Después veremos cómo esos cálculos se traducen en operaciones sobre tensores y actualizaciones de parámetros.

El objetivo será comprender PPO hasta el punto de poder implementar una versión sencilla, interpretar sus métricas y diagnosticar problemas de entrenamiento.

---

*Fin de la lección 103A.*